In [1]:
import numpy as np
from skimage import filters
import dask
import dask.array
import dask_image.ndfilters
from dask.distributed import  Client, LocalCluster
import time
memlim = '750GB' #3/4 of available RAM on machine

cluster = LocalCluster(memory_limit = memlim, n_workers=1, dashboard_address=':35000')

client = Client(cluster) # code runs just fine without calling the LocalCluster, i.e. on the threaded? scheduler

In [2]:
def Gaussian_Blur_space(da, sigma):      
    deptharray = np.ones(da.ndim)+4*sigma
    deptharray[-1] = 0
    sigmas = np.ones(deptharray.shape)*sigma
    deptharray = tuple(np.min([deptharray, da.shape], axis=0))

    sigmas[-1] = 0
    G = da.map_overlap(filters.gaussian, depth=deptharray, boundary='nearest', sigma = sigmas, preserve_range=True)
    return G

def Gaussian_Blur_space_dask(da, sigma):      
    sigmas = np.ones(da.ndim)*sigma
    sigmas[-1] = 0
    G = dask_image.ndfilters.gaussian_filter(da, mode='nearest', sigma = sigmas)
    return G

In [3]:
shp = (750, 108*10, 2016, 53*10) # typical shape of image data: (750, 108, 2016, 53)

# im = (np.random.rand(*shp)*10000).astype(np.uint16)

# da = dask.array.from_array(im, chunks='auto')
da = dask.array.random.randint(1, 16000, shp)

In [4]:
da

dask.array<randint, shape=(750, 1080, 2016, 530), dtype=int64, chunksize=(64, 64, 64, 64), chunktype=numpy.ndarray>

In [5]:
# t0 = time.time()
blur = Gaussian_Blur_space_dask(da, 1)

In [6]:
blur

dask.array<_trim, shape=(750, 1080, 2016, 530), dtype=int64, chunksize=(64, 64, 64, 64), chunktype=numpy.ndarray>

In [7]:
blur = blur.rechunk('4MiB')

In [8]:
blur = Gaussian_Blur_space(Gaussian_Blur_space_dask(blur, 3),10)

In [9]:
blur

dask.array<_trim, shape=(750, 1080, 2016, 530), dtype=float64, chunksize=(74, 92, 92, 26), chunktype=numpy.ndarray>

In [10]:
blur[:,50:300, :, :100]

dask.array<getitem, shape=(750, 250, 2016, 100), dtype=float64, chunksize=(74, 52, 92, 26), chunktype=numpy.ndarray>

In [ ]:
test = blur[:,50:300, :, :100].compute()
# print(time.time()-t0)

2023-07-26 16:55:47,630 - distributed.utils_perf - WARNING - full garbage collections took 50% CPU time recently (threshold: 10%)
2023-07-26 16:57:05,188 - distributed.utils_perf - WARNING - full garbage collections took 51% CPU time recently (threshold: 10%)
2023-07-26 16:58:39,168 - distributed.utils_perf - WARNING - full garbage collections took 52% CPU time recently (threshold: 10%)
2023-07-26 17:00:33,525 - distributed.utils_perf - WARNING - full garbage collections took 53% CPU time recently (threshold: 10%)
2023-07-26 17:02:55,408 - distributed.utils_perf - WARNING - full garbage collections took 53% CPU time recently (threshold: 10%)
/mpc/homes/fische_r/miniconda3/lib/python3.10/site-packages/distributed/client.py:3141: UserWarning: Sending large graph of size 508.95 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
2023-07-26 17:11:33,805 - distributed.utils_perf - WARNING - full garbage collections took 53% CPU time 

In [17]:
t0 = time.time()
blur_skimage = Gaussian_Blur_space(da, 1)
blur_skimage = blur_skimage.compute()
print(time.time()-t0)

2023-07-26 13:38:25,517 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2023-07-26 13:38:26,053 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2023-07-26 13:38:26,665 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2023-07-26 13:38:27,428 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2023-07-26 13:38:29,129 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2023-07-26 13:38:29,835 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2023-07-26 13:38:30,468 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2023-07-26 13:38:31,515 - distributed.utils_perf - WARNING - full garbage collections took

In [18]:
blur.max()

12971

In [19]:
blur_skimage.max()

12972.02923624857

2023-07-26 13:52:23,789 - distributed.utils_perf - WARNING - full garbage collections took 12% CPU time recently (threshold: 10%)


In [15]:
blur = blur.compute()

/mpc/homes/fische_r/miniconda3/lib/python3.10/site-packages/distributed/client.py:3141: UserWarning: Sending large graph of size 6.99 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
2023-07-26 11:29:12,673 - distributed.protocol.core - CRITICAL - Failed to Serialize
Traceback (most recent call last):
  File "/mpc/homes/fische_r/miniconda3/lib/python3.10/site-packages/distributed/protocol/core.py", line 109, in dumps
    frames[0] = msgpack.dumps(msg, default=_encode_default, use_bin_type=True)
  File "/mpc/homes/fische_r/miniconda3/lib/python3.10/site-packages/msgpack/__init__.py", line 38, in packb
    return Packer(**kwargs).pack(o)
  File "msgpack/_packer.pyx", line 294, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 300, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 297, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 264, in msgpack._cmsgpack.Packer._pack
  

CancelledError: ('_trim-f29bbfe41246458b3b3430ad0e74d59d', 4, 0, 5, 0)

In [15]:
blur = blur.compute() #warning with shp = (250, 100, 400, 50)

/mpc/homes/fische_r/miniconda3/lib/python3.10/site-packages/distributed/client.py:3141: UserWarning: Sending large graph of size 0.93 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
